# 🏛️ ĐỒ ÁN MÔN HỌC DAT301m - STAGE 1: DATA AUDITING & LABEL NOISE FILTERING
## Khử rò rỉ dữ liệu (Data Leakage), loại bỏ ảnh rác và lọc mẫu sai nhãn

* **Mục tiêu:** Dọn sạch toàn bộ dị tật cấp độ nhãn và loại bỏ rò rỉ để thước đo Validation phản ánh trung thực năng lực của mô hình.
* **Hành động ở Data:**
  1. Loại bỏ 49 ảnh rò rỉ (MD5 duplicates).
  2. Xóa ảnh trắng tinh `vi/aac26b40`.
  3. Lọc bỏ các ảnh sai nhãn (`iii/b0a13faa` là chữ X, `iii/b030ba5a` là 1 nét I, `ix/aec401ea` & `ix/aec04e56` là chữ VI).
  4. Chia lại Stratified 80/20 train/val sạch không trùng lặp.


In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("🚀 Đang chạy trên Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Đang chạy trên Local Machine")

if IN_COLAB:
    if os.path.exists("/content/deeplearningai-data-centric-competition"):
        os.chdir("/content/deeplearningai-data-centric-competition")
    elif not os.path.exists("data"):
        os.system("git clone https://github.com/Nov05/deeplearningai-data-centric-competition.git")
        if os.path.exists("deeplearningai-data-centric-competition"):
            os.chdir("deeplearningai-data-centric-competition")

# Giải nén data và label_book nếu chưa có
import tarfile
if not os.path.exists("data") and os.path.exists("original/data.tar.gz"):
    with tarfile.open("original/data.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu data thành công.")

if not os.path.exists("label_book") and os.path.exists("original/label_book.tar.gz"):
    with tarfile.open("original/label_book.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu label_book thành công.")

import glob
raw_count = len(glob.glob("data/**/*.png", recursive=True))
print(f"✅ Dữ liệu gốc: Tìm thấy {raw_count} ảnh.")
assert raw_count > 0, "LỖI: Chưa có ảnh trong thư mục 'data/'!"


🚀 Đang chạy trên Google Colab


/tmp/ipykernel_549/3864034174.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")


✅ Đã giải nén dữ liệu data thành công.
✅ Dữ liệu gốc: Tìm thấy 2880 ảnh.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import hashlib
import shutil
from PIL import Image

IMAGE_SIZE = (32, 32)
BATCH_SIZE = 32
NUM_CLASSES = 10
CLASS_NAMES = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"]
RANDOM_SEED = 123
EPOCHS = 15

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU ({gpus[0].name}) -> Tốc độ huấn luyện cực nhanh!")
else:
    print("⚠️ Thiết bị hiện tại: CPU (Khuyên dùng: Trên Colab vào Menu Runtime -> Change runtime type -> Chọn T4 GPU)")

def build_fixed_vgg16_model():
    """
    KIẾN TRÚC VGG16 TỰ XÂY DỰNG TỪ ĐẦU (FROM SCRATCH) THEO YÊU CẦU GIẢNG VIÊN:
    - KHÔNG dùng thư viện có sẵn tf.keras.applications.VGG16
    - KHÔNG dùng pretrained weights ImageNet
    - Tự code tay đúng 13 lớp Conv2D + 5 lớp MaxPooling2D + 3 lớp Dense = 16 Layers VGG16
    - Khởi tạo He-Normal, cố định bất biến qua mọi Stage
    """
    tf.random.set_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name="input_image")
    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="rescaling")(inputs)

    # Block 1 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c1")(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b1_pool")(x)

    # Block 2 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c1")(x)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b2_pool")(x)

    # Block 3 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c1")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c2")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b3_pool")(x)

    # Block 4 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b4_pool")(x)

    # Block 5 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b5_pool")(x)

    # Classifier Head (13 Conv + 3 Dense = 16 Layers VGG16)
    x = tf.keras.layers.Flatten(name="flatten")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc1")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout1")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc2")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout2")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Fixed_VGG16_Scratch")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

print("✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).")


TensorFlow Version: 2.21.0
🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU (/physical_device:GPU:0) -> Tốc độ huấn luyện cực nhanh!
✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).


## 1. THIẾT LẬP BỘ LỌC DỮ LIỆU SẠCH (DATA CLEANING PIPELINE)


In [ ]:
CLEAN_STAGE1_DIR = "stage1_cleaned_data"
if os.path.exists(CLEAN_STAGE1_DIR):
    shutil.rmtree(CLEAN_STAGE1_DIR)
os.makedirs(CLEAN_STAGE1_DIR, exist_ok=True)

KNOWN_CORRUPT_OR_MISLABELED_FILES = {
    "aac26b40-ce5d-11eb-b317-38f9d35ea60f.png", # Ảnh trắng tinh 100%
    "b0a13faa-ce5d-11eb-b317-38f9d35ea60f.png", # Chữ X nhưng gán vào iii
    "b030ba5a-ce5d-11eb-b317-38f9d35ea60f.png", # Chỉ 1 gạch dọc I nhưng gán vào iii
    "aec401ea-ce5d-11eb-b317-38f9d35ea60f.png", # Chữ VI nhưng gán vào ix
    "aec04e56-ce5d-11eb-b317-38f9d35ea60f.png", # Chữ VI nhưng gán vào ix
    "adf5ad04-ce5d-11eb-b317-38f9d35ea60f.png"  # Nét loang mực lỗi gán vào iv
}

seen_hashes = set()
total_cleaned = 0

for c in CLASS_NAMES:
    c_dest = os.path.join(CLEAN_STAGE1_DIR, c)
    os.makedirs(c_dest, exist_ok=True)
    raw_files = glob.glob(f"data/train/{c}/*.png") + glob.glob(f"data/val/{c}/*.png")

    for f in raw_files:
        fname = os.path.basename(f)
        if fname in KNOWN_CORRUPT_OR_MISLABELED_FILES:
            continue
        with open(f, "rb") as fp:
            h = hashlib.md5(fp.read()).hexdigest()
            if h in seen_hashes:
                continue
            seen_hashes.add(h)
        arr = np.array(Image.open(f).convert("L"))
        if arr.min() == 255:
            continue

        shutil.copy(f, os.path.join(c_dest, fname))
        total_cleaned += 1

print(f"✅ Đã làm sạch toàn bộ dữ liệu. Tổng số ảnh sạch: {total_cleaned} ảnh.")


✅ Đã làm sạch toàn bộ dữ liệu. Tổng số ảnh sạch: 2825 ảnh.


## 2. NẠP DỮ LIỆU SẠCH (STRATIFIED 80/20) VÀ HUẤN LUYỆN LẠI VGG16


In [ ]:
train_ds_s1 = tf.keras.preprocessing.image_dataset_from_directory(
    CLEAN_STAGE1_DIR, validation_split=0.2, subset="training",
    seed=RANDOM_SEED, labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

val_ds_s1 = tf.keras.preprocessing.image_dataset_from_directory(
    CLEAN_STAGE1_DIR, validation_split=0.2, subset="validation",
    seed=RANDOM_SEED, labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

lb_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    "label_book", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, shuffle=False,
    batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

model_s1 = build_fixed_vgg16_model()
history_s1 = model_s1.fit(
    train_ds_s1,
    validation_data=val_ds_s1,
    epochs=EPOCHS,
    verbose=1
)

val_loss_s1, val_acc_s1 = model_s1.evaluate(val_ds_s1, verbose=0)
lb_loss_s1, lb_acc_s1 = model_s1.evaluate(lb_ds_raw, verbose=0)

print("="*60)
print(f"📊 KẾT QUẢ STAGE 1 (DATA CLEANING):")
print(f"  • Validation Accuracy (Sạch không rò rỉ): {val_acc_s1*100:.2f}% | Loss: {val_loss_s1:.4f}")
print(f"  • Label Book Accuracy: {lb_acc_s1*100:.2f}% | Loss: {lb_loss_s1:.4f}")
print("="*60)


Found 2825 files belonging to 10 classes.
Using 2260 files for training.
Found 2825 files belonging to 10 classes.
Using 565 files for validation.
Found 52 files belonging to 10 classes.
Epoch 1/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 58s 465ms/step - accuracy: 0.1071 - loss: 2.3429 - val_accuracy: 0.1257 - val_loss: 2.2983
Epoch 2/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.1235 - loss: 2.3002 - val_accuracy: 0.1257 - val_loss: 2.2969
Epoch 3/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.1168 - loss: 2.2995 - val_accuracy: 0.1204 - val_loss: 2.2964
Epoch 4/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.1133 - loss: 2.2998 - val_accuracy: 0.1257 - val_loss: 2.2937
Epoch 5/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.1265 - loss: 2.2941 - val_accuracy: 0.1257 - val_loss: 2.2714
Epoch 6/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.1602 - loss: 2.2712 - val_accuracy: 0.1398 - val_loss: 2.2918
Epoch 7/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/ste